# 04 - Modelling

**Author:** Juan Ruelas  
**Date:** 2026-05

We benchmark three approaches on the 2019 test set:

1. **Seasonal naive** - predict load(t) = load(t - 168h). Hard to beat for short-horizon hourly electricity, and a fair reality check on the more elaborate models.
2. **Prophet** - additive trend + multi-period seasonality + temperature as an extra regressor. Good interpretability and built-in holidays.
3. **LightGBM** - gradient-boosted trees on the full feature matrix. Handles non-linear, interacting tabular features and is the production-grade workhorse for short-term load forecasting.

All three produce hourly predictions for 2019, stored side-by-side in `predictions.csv`.

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from prophet import Prophet

processed_dir = Path('..') / 'data' / 'processed'

train = pd.read_csv(processed_dir / 'features_train.csv', parse_dates=['timestamp'])
test  = pd.read_csv(processed_dir / 'features_test.csv',  parse_dates=['timestamp'])
print('train:', train.shape, ' test:', test.shape)

## 1. Seasonal naive baseline

For every test hour, predict the load observed exactly 168 hours earlier. Because our test set starts on 2019-01-01 and we have the entire 2018 in train, we can index back into the merged hourly history without any boundary issue.

This baseline already exploits the strongest signal in the data (weekly seasonality) and acts as a difficulty floor.

In [ ]:
history = pd.concat([train, test], ignore_index=True).sort_values('timestamp').reset_index(drop=True)
history_lookup = history.set_index('timestamp')['load_mw']

baseline_pred = test['timestamp'].map(lambda t: history_lookup.get(t - pd.Timedelta(hours=168), np.nan))
print('Baseline NaNs:', baseline_pred.isna().sum())
baseline_pred.head()

## 2. Prophet with temperature regressor

Prophet expects columns named `ds` (timestamp) and `y` (target). We disable the daily seasonality default and supply our own with `Fourier order = 10` so it can capture the sharp morning/evening peaks. Temperature is added as an extra regressor - at predict time we feed the actual test-period weather (a fair operational analogue, since day-ahead weather forecasts are extremely accurate).

In [ ]:
prophet_train = train.rename(columns={'timestamp': 'ds', 'load_mw': 'y'})[['ds', 'y', 'temperature_2m']]
prophet_test = test.rename(columns={'timestamp': 'ds'})[['ds', 'temperature_2m']]

model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
model_prophet.add_seasonality(name='daily', period=1, fourier_order=10)
model_prophet.add_regressor('temperature_2m')
model_prophet.fit(prophet_train)

prophet_forecast = model_prophet.predict(prophet_test)
prophet_pred = prophet_forecast['yhat'].values
prophet_pred[:5]

## 3. LightGBM

We treat this as a standard tabular regression. The last 20% of the (already chronologically ordered) train set acts as a validation fold for early stopping - this keeps the time order intact and avoids cross-validation leakage from lag features.

Hyperparameter choices:

- `objective='regression'` with L2 loss aligns with our RMSE evaluation.
- `num_leaves=64` and `learning_rate=0.05` are a balanced starting point; early stopping decides the actual number of trees.
- `min_data_in_leaf=50` discourages over-fitting to small calendar interactions.

In [ ]:
feature_cols = [c for c in train.columns if c not in ('timestamp', 'load_mw')]

n_train = len(train)
split_idx = int(n_train * 0.8)
X_tr, y_tr = train.iloc[:split_idx][feature_cols], train.iloc[:split_idx]['load_mw']
X_val, y_val = train.iloc[split_idx:][feature_cols], train.iloc[split_idx:]['load_mw']
X_te = test[feature_cols]

print('Train fold:', X_tr.shape, ' Val fold:', X_val.shape, ' Test:', X_te.shape)

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'verbose': -1,
}

dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

model_lgbm = lgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=200)],
)

lgbm_pred = model_lgbm.predict(X_te, num_iteration=model_lgbm.best_iteration)
print('Best iteration:', model_lgbm.best_iteration)

## 4. Persist predictions

One tidy CSV with the test timestamp, the actual load, and one column per model. Notebook 05 only needs this file.

In [ ]:
preds = pd.DataFrame({
    'timestamp': test['timestamp'].values,
    'actual':    test['load_mw'].values,
    'baseline':  baseline_pred.values,
    'prophet':   prophet_pred,
    'lgbm':      lgbm_pred,
})
preds.to_csv(processed_dir / 'predictions.csv', index=False)
print('saved predictions to', processed_dir / 'predictions.csv')
preds.head()

We also persist the LightGBM booster so notebook 05 can reuse it for residual analysis and feature-importance plots without retraining.

In [ ]:
model_lgbm.save_model(str(processed_dir / 'lgbm_model.txt'))
print('saved lgbm model.')